In [2]:
# Cell 3: Option B - HuggingFace embeddings (also free)
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings_hf = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",   # small, fast, good quality
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

test_text = "LangChain is a framework for building LLM applications"
vector_hf = embeddings_hf.embed_query(test_text)
print(f"Model: all-MiniLM-L6-v2 (HuggingFace)")
print(f"Vector dimensions: {len(vector_hf)}")
print(f"First 5 values: {vector_hf[:5]}")

C:\Users\Dhrumil Prajapati\AppData\Local\Temp\ipykernel_22432\2209561435.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_hf = HuggingFaceEmbeddings(
d:\AI\Langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3139.75it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UN

Model: all-MiniLM-L6-v2 (HuggingFace)
Vector dimensions: 384
First 5 values: [-0.015797829255461693, -0.03293266519904137, 0.006158025935292244, -0.12219040840864182, -0.0008332814322784543]


In [3]:
# Cell 4: Embedding comparison
# Which one to use?
import time

text = "What is machine learning?"

# # Time Ollama
# start = time.time()
# v1 = embeddings_ollama.embed_query(text)
# ollama_time = time.time() - start

# Time HuggingFace
start = time.time()
v2 = embeddings_hf.embed_query(text)
hf_time = time.time() - start

print("Embedding Model Comparison:")
# print(f"  nomic-embed-text (Ollama): {len(v1)} dims | {ollama_time:.3f}s")
print(f"  all-MiniLM-L6-v2 (HF):    {len(v2)} dims | {hf_time:.3f}s")
print(f"\n💡 Recommendation: Use nomic-embed-text for this course")
print(f"   Higher dimensions = better semantic understanding")

# Use Ollama embeddings for rest of notebook
embeddings = embeddings_hf

Embedding Model Comparison:
  all-MiniLM-L6-v2 (HF):    384 dims | 0.034s

💡 Recommendation: Use nomic-embed-text for this course
   Higher dimensions = better semantic understanding


In [4]:
# Cell 5: See how similarity works
import numpy as np

def cosine_similarity(v1, v2):
    """Measure similarity between two vectors (0=different, 1=identical)"""
    v1, v2 = np.array(v1), np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# Test sentences
sentences = [
    "I love programming in Python",       # base sentence
    "Python coding is my passion",        # similar meaning
    "I enjoy writing software code",      # somewhat similar
    "Machine learning is fascinating",    # different topic, tech related
    "The weather is nice today",          # completely different
    "I hate programming",                 # opposite meaning
]

base = embeddings.embed_query(sentences[0])

print(f"Base: '{sentences[0]}'")
print(f"\nSimilarity scores:")
print("-" * 60)

for sentence in sentences[1:]:
    vec = embeddings.embed_query(sentence)
    sim = cosine_similarity(base, vec)
    bar = "█" * int(sim * 30)
    print(f"  {sim:.3f} {bar}")
    print(f"         '{sentence}'")

Base: 'I love programming in Python'

Similarity scores:
------------------------------------------------------------
  0.882 ██████████████████████████
         'Python coding is my passion'
  0.637 ███████████████████
         'I enjoy writing software code'
  0.314 █████████
         'Machine learning is fascinating'
  0.080 ██
         'The weather is nice today'
  0.636 ███████████████████
         'I hate programming'


In [5]:
# Cell 6: ChromaDB setup
import chromadb
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load and split a document (from Day 6)
loader = TextLoader("sample_docs/sample.txt", encoding="utf-8")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
chunks = splitter.split_documents(documents)
print(f"✅ Prepared {len(chunks)} chunks")

✅ Prepared 4 chunks


In [6]:
# Cell 7: Create ChromaDB vector store
import shutil
import os

# Clean previous run if exists
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

# Create vector store - this embeds all chunks automatically
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",    # saves to disk!
    collection_name="langchain_docs"
)

print(f"✅ ChromaDB created!")
print(f"   Chunks stored: {vectorstore._collection.count()}")
print(f"   Location: ./chroma_db")

✅ ChromaDB created!
   Chunks stored: 4
   Location: ./chroma_db


In [7]:
# Cell 8: Query the vector store
query = "What are LangChain chains?"

results = vectorstore.similarity_search(
    query=query,
    k=3             # return top 3 most relevant chunks
)

print(f"Query: '{query}'")
print(f"Top {len(results)} results:\n")

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Content: {doc.page_content}")
    print(f"Source:  {doc.metadata.get('source', 'N/A')}")
    print()

Query: 'What are LangChain chains?'
Top 3 results:

--- Result 1 ---
Content: LangChain Framework Overview

LangChain is an open-source framework designed to simplify 
the creation of applications using large language models (LLMs).
Source:  sample_docs/sample.txt

--- Result 2 ---
Content: Chains:
Chains allow you to combine multiple components together. 
The LangChain Expression Language (LCEL) uses the pipe operator 
to create clean, readable chains.

Agents:
Agents use LLMs to decide which actions to take. They can use 
tools like web search, calculators, and custom functions.
Source:  sample_docs/sample.txt

--- Result 3 ---
Content: Core Components:
LangChain has several key components including chains, agents, 
memory, and document loaders. Each component serves a specific 
purpose in building AI applications.
Source:  sample_docs/sample.txt



In [8]:
# Cell 9: Similarity search WITH scores
results_with_scores = vectorstore.similarity_search_with_score(
    query=query,
    k=3
)

print(f"Query: '{query}'\n")
for doc, score in results_with_scores:
    print(f"Score: {score:.4f} (lower = more similar in Chroma)")
    print(f"Content: {doc.page_content[:150]}...")
    print()

Query: 'What are LangChain chains?'

Score: 0.6175 (lower = more similar in Chroma)
Content: LangChain Framework Overview

LangChain is an open-source framework designed to simplify 
the creation of applications u...

Score: 0.7278 (lower = more similar in Chroma)
Content: Chains:
Chains allow you to combine multiple components together. 
The LangChain Expression Language (LCEL) uses the pipe operator 
to create clean, r...

Score: 0.8104 (lower = more similar in Chroma)
Content: Core Components:
LangChain has several key components including chains, agents, 
memory, and document loaders. Each component serves a specific 
purpo...



In [9]:
# Cell 10: Load existing ChromaDB (persistence test)
# Reload from disk - embeddings are saved, no re-computation!
vectorstore_loaded = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
    collection_name="langchain_docs"
)

print(f"✅ Reloaded from disk!")
print(f"   Chunks: {vectorstore_loaded._collection.count()}")

# Query it - works exactly the same
results = vectorstore_loaded.similarity_search("What is memory in LangChain?", k=2)
for r in results:
    print(f"\n→ {r.page_content[:200]}")

✅ Reloaded from disk!
   Chunks: 4

→ Core Components:
LangChain has several key components including chains, agents, 
memory, and document loaders. Each component serves a specific 
purpose in building AI applications.

→ LangChain Framework Overview

LangChain is an open-source framework designed to simplify 
the creation of applications using large language models (LLMs).


In [10]:
# Cell 11: FAISS - faster for search, no persistence by default
from langchain_community.vectorstores import FAISS

# Create FAISS store
faiss_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("✅ FAISS store created (in memory)")

# Search
results = faiss_store.similarity_search("How do agents work?", k=3)
print(f"\nFAISS results for 'How do agents work?':")
for i, r in enumerate(results):
    print(f"\n{i+1}. {r.page_content[:200]}")

✅ FAISS store created (in memory)

FAISS results for 'How do agents work?':

1. Chains:
Chains allow you to combine multiple components together. 
The LangChain Expression Language (LCEL) uses the pipe operator 
to create clean, readable chains.

Agents:
Agents use LLMs to decide

2. Core Components:
LangChain has several key components including chains, agents, 
memory, and document loaders. Each component serves a specific 
purpose in building AI applications.

3. Memory:
Memory allows chatbots to remember previous conversations. 
This is crucial for building coherent multi-turn applications.

RAG (Retrieval Augmented Generation):
RAG combines document retrieva


In [11]:
# Cell 12: Save and load FAISS
faiss_store.save_local("./faiss_db")
print("✅ FAISS saved to disk")

# Load it back
faiss_loaded = FAISS.load_local(
    "./faiss_db",
    embeddings,
    allow_dangerous_deserialization=True  # required flag
)
print("✅ FAISS loaded from disk")
print(f"   Works: {len(faiss_loaded.similarity_search('test', k=1))} result")

✅ FAISS saved to disk
✅ FAISS loaded from disk
   Works: 1 result


In [12]:
# Cell 13: Head to head comparison
import time

queries = [
    "What is LangChain?",
    "How does memory work?",
    "What are chains used for?"
]

print("ChromaDB vs FAISS comparison")
print("="*50)

for query in queries:
    # ChromaDB
    start = time.time()
    chroma_results = vectorstore.similarity_search(query, k=2)
    chroma_time = time.time() - start

    # FAISS
    start = time.time()
    faiss_results = faiss_store.similarity_search(query, k=2)
    faiss_time = time.time() - start

    print(f"\nQuery: '{query}'")
    print(f"  ChromaDB: {chroma_time*1000:.1f}ms")
    print(f"  FAISS:    {faiss_time*1000:.1f}ms")
    print(f"  Same top result? {chroma_results[0].page_content[:50] == faiss_results[0].page_content[:50]}")

ChromaDB vs FAISS comparison

Query: 'What is LangChain?'
  ChromaDB: 30.9ms
  FAISS:    27.7ms
  Same top result? True

Query: 'How does memory work?'
  ChromaDB: 21.7ms
  FAISS:    18.7ms
  Same top result? True

Query: 'What are chains used for?'
  ChromaDB: 25.3ms
  FAISS:    22.2ms
  Same top result? True


When to use which:
┌─────────────────────────────────────────────────────┐
│  ChromaDB  → persistent, filterable, production     │
│              great for growing document collections │
│                                                     │
│  FAISS     → blazing fast search, in-memory         │
│              great for fixed datasets, prototyping  │
└─────────────────────────────────────────────────────┘